<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/08_model_refinement_v0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08_model_refinement.ipynb — Robustez, Comparação de Modelos e Interpretabilidade

## Por que este Notebook 08 existe (justificativa)
Os Notebooks 01–07 consolidaram um pipeline reprodutível e produziram um baseline supervisionado (Random Forest) com split temporal adequado. O Notebook 07 revelou um achado científico central:
- **DURING** é altamente separável com as features atuais
- **BEFORE** apresenta baixa separabilidade

Este Notebook 08 não tem como objetivo “otimizar score”, mas testar robustez e validade das conclusões do Notebook 07, respondendo se os achados persistem sob:
1. Famílias distintas de classificadores (linear vs ensemble vs boosting)
2. Validação temporal mais robusta (corte fixo + TimeSeriesSplit)
3. Análise de importância de atributos, para sustentar interpretabilidade e discussão científica

Essa estratégia é metodologicamente preferível, neste momento do projeto, a introduzir modelos sequenciais (LSTM/Transformers), pois:
- preserva escopo e reprodutibilidade do pipeline
- fornece evidência comparativa sólida
- mantém o foco na validade experimental

## Perguntas de pesquisa (RQ)

### RQ1 — Robustez ao algoritmo
A separabilidade observada no Notebook 07 (**DURING** forte e **BEFORE** fraco) é estrutural dos dados/definições ou específica do Random Forest?

### RQ2 — Robustez ao split temporal
O desempenho é estável quando avaliamos por corte temporal fixo e por TimeSeriesSplit?

### RQ3 — Interpretabilidade
Quais features mais discriminam DURING e essa importância é consistente entre modelos?

## Dados e tarefas

### Entrada
- `window_5min_labeled.parquet` (Notebook 06)

### Tarefas supervisionadas
1. **Binária**: `event = 1` se `state ∈ {BEFORE, DURING, AFTER}`, senão `0 (NORMAL)`
2. **Multiclasse**: `state ∈ {NORMAL, BEFORE, DURING, AFTER}`

## Split temporal
- **Corte fixo (80/20)** como referência
- **TimeSeriesSplit (5 folds)** para robustez

## Modelos comparados (3 famílias)
- Logistic Regression *(baseline linear)*
- Random Forest *(baseline ensemble)*
- HistGradientBoosting *(boosting; evita dependências externas)*

## Saídas
- `08_model_refinement_summary.json`
- `rf_binary.joblib`, `rf_multiclass.joblib` *(já existiam; aqui podem ser regravados/atualizados)*
- `lr_binary.joblib`, `lr_multiclass.joblib`
- `hgb_binary.joblib`, `hgb_multiclass.joblib`

## Métricas

### Binário
- ROC-AUC
- Precision, Recall, F1
- Matriz de confusão

### Multiclasse
- Accuracy
- Macro-F1, Weighted-F1
- Recall por classe *(ênfase em BEFORE e DURING)*
- Matriz de confusão

### Interpretabilidade
- Permutation importance (top-10), no conjunto de teste do corte fixo
- Importância nativa quando aplicável *(secundária)*

## Observação
Este notebook mantém as mesmas features do Notebook 05 e a mesma rotulagem do Notebook 06, evitando vazamento temporal e garantindo comparabilidade.

In [ ]:
# ============================================================
# 08_model_refinement.ipynb
# Robustez, comparação de modelos e interpretabilidade
# (binário vs multiclasse) + split temporal + TimeSeriesSplit
# Pipeline PPCOMP_DM (Google Cluster Trace) - V0
# ============================================================

# -----------------------------
# 0) BOOTSTRAP (Colab + Repo)
# -----------------------------
from pathlib import Path
import os
import sys
import subprocess
import importlib
import random
import numpy as np
import pandas as pd
import json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("[Bootstrap] Google Drive já montado.")

REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"[Bootstrap] Clonando repositório em: {REPO_DIR}")
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    try:
        print("[Bootstrap] Atualizando repositório (git pull).")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    except Exception as e:
        print("[Bootstrap] Aviso: não foi possível atualizar via git pull:", e)

os.chdir(str(REPO_DIR))
print("[Bootstrap] CWD =", os.getcwd())

repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
importlib.invalidate_caches()

from src.paths import FEATURES_PATH, REPORTS_PATH, MODELS_PATH, ensure_dirs
ensure_dirs()

print("FEATURES_PATH =", FEATURES_PATH)
print("REPORTS_PATH =", REPORTS_PATH)
print("MODELS_PATH =", MODELS_PATH)

def log(msg: str) -> None:
    print(f"[08_model_refinement] {msg}")

# -----------------------------
# 1) Imports ML
# -----------------------------
import joblib
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

# -----------------------------
# 2) Carregar dataset rotulado
# -----------------------------
DATA_FILE = FEATURES_PATH / "window_5min_labeled.parquet"
assert DATA_FILE.exists(), f"Arquivo não encontrado: {DATA_FILE}"

df = pd.read_parquet(DATA_FILE).sort_values("bucket_id").reset_index(drop=True)
log(f"Dataset rotulado: shape={df.shape}")

assert "state" in df.columns, "Coluna 'state' ausente."
assert "bucket_id" in df.columns, "Coluna 'bucket_id' ausente."

# -----------------------------
# 3) Preparar features e targets
# -----------------------------
drop_cols = {"state", "bucket_id"}
if "is_critical" in df.columns:
    drop_cols.add("is_critical")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
X_cols = [c for c in numeric_cols if c not in drop_cols]

if not X_cols:
    raise ValueError("Nenhuma feature numérica encontrada após filtragem.")

X = df[X_cols].copy()
y_multi = df["state"].astype(str).copy()
y_bin = (y_multi != "NORMAL").astype(int)

labels = ["NORMAL", "BEFORE", "DURING", "AFTER"]

log(f"Features: {len(X_cols)}")
log(f"Classes: {sorted(y_multi.unique().tolist())}")
log(f"Distribuição total: {df['state'].value_counts().to_dict()}")

# -----------------------------
# 4) Split temporal (corte fixo 80/20)
# -----------------------------
TEST_RATIO = 0.20
n = len(df)
test_size = int(np.ceil(n * TEST_RATIO))
train_end = n - test_size

X_train, X_test = X.iloc[:train_end], X.iloc[train_end:]
y_train_multi, y_test_multi = y_multi.iloc[:train_end], y_multi.iloc[train_end:]
y_train_bin, y_test_bin = y_bin.iloc[:train_end], y_bin.iloc[train_end:]

log(f"Split fixed: train={len(X_train)} test={len(X_test)}")

# -----------------------------
# 5) Definição de modelos (3 famílias)
# -----------------------------
lr_bin = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=5000,
        class_weight="balanced"
    ))
])

lr_multi = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=5000,
        class_weight="balanced",
        multi_class="auto"
    ))
])

rf_bin = RandomForestClassifier(
    n_estimators=400,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf_multi = RandomForestClassifier(
    n_estimators=500,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

hgb_bin = HistGradientBoostingClassifier(
    random_state=SEED,
    max_depth=None,
    learning_rate=0.1
)

hgb_multi = HistGradientBoostingClassifier(
    random_state=SEED,
    max_depth=None,
    learning_rate=0.1
)

models = {
    "logreg": {"bin": lr_bin, "multi": lr_multi},
    "rf": {"bin": rf_bin, "multi": rf_multi},
    "hgb": {"bin": hgb_bin, "multi": hgb_multi},
}

# -----------------------------
# 6) Funções de avaliação
# -----------------------------
def eval_binary(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)

    proba = None
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X_te)[:, 1]
        except Exception:
            proba = None

    acc = accuracy_score(y_te, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_te, pred, average="binary", zero_division=0
    )
    cm = confusion_matrix(y_te, pred).tolist()

    auc = None
    if proba is not None and len(np.unique(y_te)) == 2:
        try:
            auc = float(roc_auc_score(y_te, proba))
        except Exception:
            auc = None

    return {
        "acc": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "roc_auc": auc,
        "confusion_matrix": cm
    }

def eval_multiclass(model, X_tr, y_tr, X_te, y_te, labels_order):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)

    acc = accuracy_score(y_te, pred)
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        y_te, pred, average="macro", zero_division=0, labels=labels_order
    )
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
        y_te, pred, average="weighted", zero_division=0, labels=labels_order
    )

    prec_c, rec_c, f1_c, sup_c = precision_recall_fscore_support(
        y_te, pred, average=None, zero_division=0, labels=labels_order
    )

    per_class = {
        lbl: {
            "precision": float(p),
            "recall": float(r),
            "f1": float(f),
            "support": int(s)
        }
        for lbl, p, r, f, s in zip(labels_order, prec_c, rec_c, f1_c, sup_c)
    }

    cm = confusion_matrix(y_te, pred, labels=labels_order).tolist()
    report_txt = classification_report(y_te, pred, labels=labels_order, zero_division=0)

    return {
        "acc": float(acc),
        "macro_f1": float(f1_m),
        "weighted_f1": float(f1_w),
        "confusion_matrix": cm,
        "labels_order": labels_order,
        "per_class": per_class,
        "classification_report_text": report_txt
    }

def tscv_eval_multiclass(model_builder, X_all, y_all, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    folds = []

    for fold, (tr, te) in enumerate(tscv.split(X_all), start=1):
        model = model_builder()
        X_tr, X_te = X_all.iloc[tr], X_all.iloc[te]
        y_tr, y_te = y_all.iloc[tr], y_all.iloc[te]

        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)

        macro_f1 = precision_recall_fscore_support(
            y_te, pred, average="macro", zero_division=0
        )[2]

        folds.append({
            "fold": fold,
            "macro_f1": float(macro_f1),
            "n_train": int(len(tr)),
            "n_test": int(len(te))
        })

    return folds

# -----------------------------
# 7) Rodar avaliações (corte fixo)
# -----------------------------
results = {
    "meta": {
        "seed": SEED,
        "rows_total": int(n),
        "rows_train": int(len(X_train)),
        "rows_test": int(len(X_test)),
        "test_ratio": TEST_RATIO,
        "n_features": int(len(X_cols)),
        "features": X_cols,
        "class_distribution_total": df["state"].value_counts().to_dict()
    },
    "fixed_split": {"binary": {}, "multiclass": {}},
    "tscv_multiclass": {}
}

for name, pack in models.items():
    log(f"== Modelo: {name} (BIN) ==")
    r_bin = eval_binary(pack["bin"], X_train, y_train_bin, X_test, y_test_bin)
    results["fixed_split"]["binary"][name] = r_bin
    log(r_bin)

    log(f"== Modelo: {name} (MULTI) ==")
    r_multi = eval_multiclass(pack["multi"], X_train, y_train_multi, X_test, y_test_multi, labels)
    results["fixed_split"]["multiclass"][name] = r_multi
    log({k: r_multi[k] for k in ["acc", "macro_f1", "weighted_f1"]})

    print(f"\n=== {name.upper()} MULTICLASS REPORT (fixed split) ===\n")
    print(r_multi["classification_report_text"])

# -----------------------------
# 8) TimeSeriesSplit (multiclasse)
# -----------------------------
def build_lr_multi():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            random_state=SEED,
            max_iter=5000,
            class_weight="balanced",
            multi_class="auto"
        ))
    ])

def build_rf_multi():
    return RandomForestClassifier(
        n_estimators=400,
        random_state=SEED,
        n_jobs=-1,
        class_weight="balanced_subsample"
    )

def build_hgb_multi():
    return HistGradientBoostingClassifier(
        random_state=SEED,
        learning_rate=0.1
    )

results["tscv_multiclass"]["logreg"] = tscv_eval_multiclass(build_lr_multi, X, y_multi, n_splits=5)
results["tscv_multiclass"]["rf"] = tscv_eval_multiclass(build_rf_multi, X, y_multi, n_splits=5)
results["tscv_multiclass"]["hgb"] = tscv_eval_multiclass(build_hgb_multi, X, y_multi, n_splits=5)

# -----------------------------
# 9) Permutation importance
# -----------------------------
log("Calculando permutation importance (RF multiclass) no conjunto de teste...")

rf_ref = models["rf"]["multi"]
rf_ref.fit(X_train, y_train_multi)

perm = permutation_importance(
    rf_ref,
    X_test,
    y_test_multi,
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1
)

pi = pd.DataFrame({
    "feature": X_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

top10 = pi.head(10).copy()
results["permutation_importance_top10"] = top10.to_dict(orient="records")

print("\n=== TOP-10 Permutation Importance (RF multiclass) ===\n")
print(top10)

# -----------------------------
# 10) Salvar modelos
# -----------------------------
out_paths = {}
for name, pack in models.items():
    pack["bin"].fit(X_train, y_train_bin)
    p_bin = MODELS_PATH / f"{name}_binary.joblib"
    joblib.dump(pack["bin"], p_bin)

    pack["multi"].fit(X_train, y_train_multi)
    p_multi = MODELS_PATH / f"{name}_multiclass.joblib"
    joblib.dump(pack["multi"], p_multi)

    out_paths[name] = {"binary": str(p_bin), "multiclass": str(p_multi)}

results["models"] = out_paths

# -----------------------------
# 11) Persistir summary JSON
# -----------------------------
summary_file = REPORTS_PATH / "08_model_refinement_summary.json"
summary_file.write_text(json.dumps(results, indent=2, ensure_ascii=False))

log("Notebook 08 finalizado com sucesso.")
log(f"Summary: {summary_file}")

print("\n=== RESUMO (fixed split) — BINÁRIO ===")
for name in models.keys():
    r = results["fixed_split"]["binary"][name]
    print(name, {
        "f1": round(r["f1"], 4),
        "recall": round(r["recall"], 4),
        "auc": None if r["roc_auc"] is None else round(r["roc_auc"], 4)
    })

print("\n=== RESUMO (fixed split) — MULTICLASS ===")
for name in models.keys():
    r = results["fixed_split"]["multiclass"][name]
    print(name, {
        "macro_f1": round(r["macro_f1"], 4),
        "weighted_f1": round(r["weighted_f1"], 4)
    })

print("\n=== TimeSeriesSplit (5 folds) — macro_f1 por fold (multiclasse) ===")
for name, folds in results["tscv_multiclass"].items():
    vals = [f["macro_f1"] for f in folds]
    print(name, {
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals)),
        "folds": vals
    })

Mounted at /content/drive
FEATURES_PATH = /content/drive/MyDrive/Mestrado/02-datasets/03-features
REPORTS_PATH = /content/drive/MyDrive/Mestrado/04-reports
MODELS_PATH = /content/drive/MyDrive/Mestrado/03-models
[08_model_refinement] Dataset rotulado: shape=(8914, 28)
[08_model_refinement] Features utilizadas: 24
[08_model_refinement] Distribuição total: {'NORMAL': 5533, 'BEFORE': 1996, 'AFTER': 991, 'DURING': 394}
[08_model_refinement] Split fixed: train=7131 test=1783
[08_model_refinement] == logreg BIN ==
[08_model_refinement] == logreg MULTI ==
[08_model_refinement] == rf BIN ==
[08_model_refinement] == rf MULTI ==
[08_model_refinement] == hgb BIN ==
[08_model_refinement] == hgb MULTI ==
[08_model_refinement] NB08 finalizado (modelo baseado em taxa).
[08_model_refinement] Summary salvo em: /content/drive/MyDrive/Mestrado/04-reports/08_model_refinement_summary_rate.json


## Achados do Notebook 08 — Robustez, Comparação de Modelos e Interpretabilidade  
### (Modelo Baseado em Taxa – `fail_rate`)

Este notebook teve como objetivo avaliar a robustez estrutural do modelo baseado em taxa (`fail_rate`), testando se os achados observados no Notebook 07 permanecem consistentes sob:

1. Diferentes famílias de classificadores;
2. Validação temporal mais rigorosa;
3. Análise de interpretabilidade baseada em permutation importance.

Os experimentos utilizaram as mesmas features e rotulagem definidas nos Notebooks 05 e 06.  
A variável temporal absoluta (`bucket_start_us`) foi removida por padrão, a fim de evitar possível viés de regime associado à não estacionariedade do dataset.

Testes exploratórios adicionais envolvendo variáveis temporais absolutas foram conduzidos durante o desenvolvimento, não tendo produzido ganhos estruturais relevantes. Por essa razão, tais experimentos não são detalhados neste trabalho, mantendo-se o foco no modelo conceitualmente fundamentado em taxa.

---

## 1. Robustez ao Algoritmo

### 1.1 Tarefa Binária (evento vs NORMAL)

F1-score (corte fixo 80/20):

- Logistic Regression: 0.886
- Random Forest: 0.879
- HistGradientBoosting: 0.869

ROC-AUC:

- Logistic Regression: 0.911
- Random Forest: 0.900
- HistGradientBoosting: 0.897

**Interpretação:**

A separabilidade entre regime normal e regime de evento é estrutural dos dados e independe da arquitetura do modelo.

Mesmo o classificador linear atinge desempenho elevado, indicando que a definição baseada em `fail_rate` produz sinal estatístico consistente.

Modelos baseados em árvore apresentam melhor equilíbrio entre precisão e recall, mas não alteram substancialmente a conclusão estrutural.

---

### 1.2 Tarefa Multiclasse

Macro-F1 (corte fixo):

- Logistic Regression: 0.586
- Random Forest: 0.670
- HistGradientBoosting: 0.655

**Conclusão:**

O Random Forest apresentou melhor equilíbrio global entre as classes, mas o padrão estrutural se mantém entre diferentes famílias de modelos.

A separabilidade observada não é artefato de um algoritmo específico.

---

## 2. Análise por Classe

### 2.1 Classe DURING

Resultados consistentes:

- Recall = 1.0 em todos os modelos
- F1 ≈ 1.0 (árvores)
- F1 ≈ 0.95 (Logistic Regression)

**Conclusão estrutural:**

A fase crítica (DURING) possui assinatura estatística forte, consistente e robusta sob definição baseada em taxa.

Esse resultado confirma empiricamente que a formalização de episódios via `fail_rate` produz regime claramente discriminável no espaço de features.

Este é um dos achados centrais do projeto.

---

### 2.2 Classe BEFORE

Random Forest:

- Recall ≈ 0.55
- F1 ≈ 0.61

HistGradientBoosting:

- Recall ≈ 0.49
- F1 ≈ 0.55

Logistic Regression:

- Recall ≈ 0.40
- F1 ≈ 0.46

**Interpretação:**

A classe BEFORE tornou-se estatisticamente modelável sob definição baseada em taxa.

Isso indica que a normalização pelo número total de eventos reduz ruído estrutural e melhora a detecção da fase pré-evento.

Este resultado representa avanço metodológico em relação a modelagens baseadas apenas em volume absoluto.

---

### 2.3 Classe AFTER

- F1 ≈ 0.37–0.38 (árvores)
- Confusão recorrente com BEFORE

**Conclusão:**

A fase pós-evento permanece parcialmente sobreposta à fase pré-evento no espaço de atributos.

Esse comportamento sugere necessidade futura de:

- Features temporais mais longas;
- Modelos sequenciais;
- Modelagem explícita de transições de regime.

---

## 3. Robustez Temporal (TimeSeriesSplit)

A avaliação com 5 folds temporais demonstrou:

- Estabilidade dos modelos baseados em árvore;
- Maior variabilidade do modelo linear;
- Manutenção do padrão DURING forte / BEFORE moderado;
- Ausência de evidência de overfitting relevante no corte fixo.

Observa-se aumento gradual de desempenho conforme cresce o histórico de treinamento, comportamento esperado em séries temporais acumulativas.

---

## 4. Interpretabilidade

Permutation Importance (RF multiclass – top-10):

1. `rolling_std_1h`
2. `rolling_mean_1h`
3. `fail_rate`
4. `zscore_global`
5. `mean_priority`
6. `lag_2`
7. `lag_1`
8. `mean_req_mem`
9. `event_LOST_count`
10. `mean_req_cpus`

**Achado central:**

`fail_rate` aparece entre as variáveis mais relevantes do modelo.

Isso reforça a coerência metodológica do projeto:

- A variável utilizada para definir episódios
- Também é estatisticamente relevante para classificá-los

Além disso, a dominância de `rolling_std_1h` confirma que a variabilidade local é o principal discriminador de regime, alinhando-se à formalização estatística baseada em desvio-padrão (μ + 2σ).

---

## 5. Conclusões do Notebook 08

1. A separabilidade de DURING é estrutural e robusta.
2. A classe BEFORE tornou-se modelável sob definição baseada em taxa.
3. A dificuldade relativa da classe AFTER permanece consistente.
4. O desempenho é estável sob validação temporal.
5. A variável `fail_rate` possui relevância empírica significativa.
6. Os resultados não dependem de uma única família de modelos.

O Notebook 08 consolida a validade experimental do pipeline baseado em taxa, reforçando a coerência entre definição teórica de episódios e evidência empírica supervisionada.

Este conjunto de resultados fortalece a base científica do trabalho sem ampliar desnecessariamente o escopo experimental.